In [31]:
!pip3 install lxml
!pip install requests lxml
!pip install selenium

In [32]:
import requests
import re
import csv
import pandas as pd
import lxml.html as l
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from collections import defaultdict

In [46]:
source = "https://www.brickeconomy.com"

themes = [
    "star-wars",
    "harry-potter",
    "speed-champions",
    "technic",
    "city",
]

list_links_sets = []

for theme in themes:
  for i in range(1, 200):
      url = f"{source}/sets/theme/{theme}?page={i}"
      get_link = requests.get(url)
      soup = BeautifulSoup(get_link.text, "html.parser")
      list_links = soup.find_all("a", href=True)

      j = 0
      for k in list_links:
          href = k["href"]
          if k["href"].startswith("/set/") and source + k["href"] not in list_links_sets:
              val = source + href
              list_links_sets.append(val)
              j = j + 1

/set/SID0036040-1/star-wars-han-solo-indiana-jones-transformation-chamber
/set/10123-1/lego-star-wars-cloud-city
/set/SID0037140-1/lego-star-wars-celebration-iv-exclusive
/set/75419-1/lego-star-wars-ucs-death-star
/set/75345-1/star-wars-the-clone-wars-501st-clone-troopers-battle-pack
/set/75409-1/lego-star-wars-jango-fetts-firespray-class-starship
/set/75417-1/lego-star-wars-at-st-walker
/set/66808-1/lego-star-wars-epic-battle-set
/set/75428-1/lego-star-wars-battle-droid-with-stap
/set/75403-1/lego-star-wars-the-mandalorian-grogu-with-hover-pram
/set/75416-1/lego-star-wars-ahsoka-chopper-c1-10p-astromech-droid
/set/75430-1/lego-star-wars-wicket-the-ewok
/set/75435-1/star-wars-the-clone-wars-battle-of-felucia-separatist-mtt
/set/75434-1/lego-star-wars-andor-k-2so-security-droid
/set/75413-1/lego-star-wars-the-clone-wars-republic-juggernaut
/set/75429-1/lego-star-wars-helmet-collection-at-at-driver-helmet
/set/75433-1/lego-star-wars-jango-fetts-starship
/set/75407-1/lego-brick-built-star

In [50]:
config = Options()
config.add_argument("--headless=new")
config.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=config)
driver.set_window_size(1920, 1080)
html_pages = []

for link in list_links_sets[:1000]:
  driver.get(link)
  val_pages = driver.page_source
  html_pages.append(val_pages)
driver.quit()

In [171]:
def repair_minifigures(val: str) -> int:
    if not val:
        return 0

    m = re.search(r'\d+', val)
    return int(m.group()) if m else 0

def repair_Retail_price(valOne: str):
    valOne = valOne.replace(",", ".")
    valTwo = re.search(r"\d+(\.\d+)?", valOne)
    return float(valTwo.group()) if valTwo else None


In [176]:
all_sets = []

for page in html_pages[1:]:
  root = l.fromstring(page)
  name = root.xpath("//h1[@class='setheader']")
  theme = root.xpath("//div[@class='col-xs-7']//a[contains(@href,'/sets/theme')]")
  subtheme = root.xpath("//div[@class='col-xs-7']//a[contains(@href,'/sets/theme/star-wars/subtheme')]")
  year = root.xpath("//div[@class='col-xs-7']//a[contains(@href,'/sets/year/')]")
  rowlists = root.xpath("//div[@class='row rowlist']")
  dict_sets = {}

  if int(year[0].text_content()) <= 2022:
    record = {}
    for row in rowlists:
        key = row.xpath(".//div[@class='col-xs-5 text-muted']")
        val = row.xpath(".//div[@class='col-xs-7']")

        if key and val:
            k = key[0].text_content().strip()
            v = val[0].text_content().strip()
            record[k] = v

    pieces_raw = record.get("Pieces", "0")
    pieces_clean = pieces_raw.split("(")[0].strip()
    pieces_clean = pieces_clean.replace(",", "").replace("\xa0", "")

    record["Theme"] = record.get("Theme", "0")
    record["Subtheme"] = record.get("Subtheme", "0")
    record["Pieces"] = record.get("Pieces", "0")
    record["Pieces"] = int(pieces_clean)
    record["Minifigs"] = repair_minifigures(record.get("Minifigs", "0"))
    record["Year"] = repair_minifigures(record.get("Year", "0"))
    record["Retail price"] = repair_Retail_price(record.get("Retail price", "0"))

    if record.get("Subtheme", 0) == 0:
      dict_sets["subtheme"] = ""

    dict_sets["name"] = name[0].text_content()
    dict_sets["theme"] = theme[0].text_content()
    dict_sets["Subtheme"] = record["Subtheme"]
    dict_sets["Pieces"] = record["Pieces"]
    dict_sets["minifigs"] = record["Minifigs"]
    dict_sets["Year"] = record["Year"]
    dict_sets["Retail price"] = record["Retail price"]

    sums = defaultdict(float)
    counts = defaultdict(int)
    div_pricechart = root.xpath("//div[@id='pricechart']")
    tr_list = div_pricechart[0].xpath(".//table//tr")

    for tr in tr_list[1:]:
      td_date = tr.xpath(".//td")[0].text_content()
      td_price = tr.xpath(".//td")[1].text_content().replace(",","")
      td_date_year = td_date.split(', ')[-1]

      if int(td_date_year) <= 2025:
        sums[td_date_year] += float(td_price)
        counts[td_date_year] += 1
        dict_sets["avg_year"] = td_date_year

    for i in sums:
      avg_price = round(float(sums[i] / counts[i]), 2)
      dict_sets["avg_price"] = avg_price
      dict_sets["avg_year"] = i
      all_sets.append(dict(dict_sets))

In [186]:
df = pd.DataFrame(all_sets)
df.head(50)

,name,theme,Subtheme,Pieces,minifigs,Year,Retail price,avg_year,avg_price
0,10123 LEGO Star Wars Cloud City,Star Wars,Episode V,698,7,2003,99.99,2015,99.99
1,10123 LEGO Star Wars Cloud City,Star Wars,Episode V,698,7,2003,99.99,2016,1481.68
2,10123 LEGO Star Wars Cloud City,Star Wars,Episode V,698,7,2003,99.99,2017,1281.40
3,10123 LEGO Star Wars Cloud City,Star Wars,Episode V,698,7,2003,99.99,2018,1319.69
4,10123 LEGO Star Wars Cloud City,Star Wars,Episode V,698,7,2003,99.99,2019,1351.93
5,10123 LEGO Star Wars Cloud City,Star Wars,Episode V,698,7,2003,99.99,2020,1728.84
6,10123 LEGO Star Wars Cloud City,Star Wars,Episode V,698,7,2003,99.99,2021,3289.34
7,10123 LEGO Star Wars Cloud City,Star Wars,Episode V,698,7,2003,99.99,2022,6602.94
8,10123 LEGO Star Wars Cloud City,Star Wars,Episode V,698,7,2003,99.99,2023,8463.12
9,10123 LEGO Star Wars Cloud City,Star Wars,Episode V,698,7,2003,99.99,2024,8576.35


In [189]:
df.to_csv("lego-sets.csv", sep=";", index=False, encoding="utf-8")

Таргет: предсказать среднюю рыночную цену LEGO - набора через 3 года.